### Simulation: Equal ETF Contribution Timing Strategy

**Portfolio:** 33.3% SPY, 33.3% QQQ, 33.3% VTI <br>
**Strategy:** Buy shares on day 1, sells shares when moving average falls below the current ETF price, then buy more shares with stored cash when moving average is greater than the current price <br>
**Initial Investment:** $10,000 <br>
**Biweekly Contribution:** Add $500 every 10 trading days and buy more shares. <br>

#### 1. Import Libaries and Load Dataset

In [16]:
import pandas as pd
import os 

PROCESSED_DIR   = "../data/processed"
SIMULATIONS_DIR = "../data/simulations"
os.makedirs(SIMULATIONS_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(PROCESSED_DIR, "prices_with_indicators.csv"))
df.head()

,Date,SPY_Price,QQQ_Price,VTI_Price,SPY_MA50,QQQ_MA50,VTI_MA50
0,2001-06-15,77.995598,35.981022,36.027668,NaN,NaN,NaN
1,2001-06-18,77.617950,35.499580,35.797894,NaN,NaN,NaN
2,2001-06-19,77.957207,35.322208,35.898216,NaN,NaN,NaN
3,2001-06-20,78.366867,36.124603,36.276844,NaN,NaN,NaN
4,2001-06-21,79.256599,36.614502,36.568089,NaN,NaN,NaN


In [17]:
# columns that should be numeric
numeric_cols = [
    "SPY_Price", "QQQ_Price", "VTI_Price",
    "SPY_MA50", "QQQ_MA50", "VTI_MA50"
]

for col in numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("`", "", regex=False)   # remove backticks
        .str.replace("'", "", regex=False)   # remove apostrophes too, just in case
        .str.replace(",", "", regex=False)   # remove commas if any
        .str.strip()
    )

    # convert cleaned text back to numeric
    df[col] = pd.to_numeric(df[col], errors="coerce")

### 2. Initialize Variables

In [19]:
initial_investment = 10000  #Initial investment of $10,000
biweekly_contribution = 500   #Biweekly contribution of $500

#portfolio weights for SPY, QQQ, and VTI
weights = {
    "SPY": 1/3,
    "QQQ": 1/3,
    "VTI": 1/3
}

#Invest everything on day 1
first_spy_price = df.loc[0, "SPY_Price"]
first_qqq_price = df.loc[0, "QQQ_Price"]
first_vti_price = df.loc[0, "VTI_Price"]

#buy starting shares based on the initial investment and weights
shares_spy = (initial_investment * weights["SPY"]) / first_spy_price
shares_qqq = (initial_investment * weights["QQQ"]) / first_qqq_price
shares_vti = (initial_investment * weights["VTI"]) / first_vti_price

cash = 0
total_invested = initial_investment

rows = []
previous_portfolio_value = None

### 3. Start Simulation

In [20]:
for i in range(len(df)):
    
    source_row = df.loc[i]
    
    date = source_row["Date"]
    
    spy_price = source_row["SPY_Price"]
    qqq_price = source_row["QQQ_Price"]
    vti_price = source_row["VTI_Price"]
    
    spy_ma50 = source_row["SPY_MA50"]
    qqq_ma50 = source_row["QQQ_MA50"]
    vti_ma50 = source_row["VTI_MA50"]
    
    contribution = 0
    
    position_spy = 0
    position_qqq = 0
    position_vti = 0
    
    #Biweekly contribution every 10 trading days
    if i % 10 == 0 and i != 0:
        contribution = biweekly_contribution
        cash += contribution
        total_invested += contribution
    
    # -------- SPY --------
    if pd.notna(spy_ma50):
        if spy_price > spy_ma50:
            position_spy = 1
        else:
            position_spy = 0
            if shares_spy > 0:
                cash += shares_spy * spy_price
                shares_spy = 0
    else:
        position_spy = 1 if shares_spy > 0 else 0
    
    # -------- QQQ --------
    if pd.notna(qqq_ma50):
        if qqq_price > qqq_ma50:
            position_qqq = 1
        else:
            position_qqq = 0
            if shares_qqq > 0:
                cash += shares_qqq * qqq_price
                shares_qqq = 0
    else:
        position_qqq = 1 if shares_qqq > 0 else 0
    
    # -------- VTI --------
    if pd.notna(vti_ma50):
        if vti_price > vti_ma50:
            position_vti = 1
        else:
            position_vti = 0
            if shares_vti > 0:
                cash += shares_vti * vti_price
                shares_vti = 0
    else:
        position_vti = 1 if shares_vti > 0 else 0
    
    #Invest cash according to weights and ETFs in position to be bought
    if cash > 0:
        investable_weights = {}

        if position_spy == 1:
            investable_weights["SPY"] = weights["SPY"]
        if position_qqq == 1:
            investable_weights["QQQ"] = weights["QQQ"]
        if position_vti == 1:
            investable_weights["VTI"] = weights["VTI"]

        total_active_weight = sum(investable_weights.values())

        if total_active_weight > 0:
            available_cash = cash
            total_invested_today = 0

            if "SPY" in investable_weights:
                amount_spy = available_cash * (investable_weights["SPY"] / total_active_weight)
                shares_spy += amount_spy / spy_price
                total_invested_today += amount_spy

            if "QQQ" in investable_weights:
                amount_qqq = available_cash * (investable_weights["QQQ"] / total_active_weight)
                shares_qqq += amount_qqq / qqq_price
                total_invested_today += amount_qqq

            if "VTI" in investable_weights:
                amount_vti = available_cash * (investable_weights["VTI"] / total_active_weight)
                shares_vti += amount_vti / vti_price
                total_invested_today += amount_vti

            cash -= total_invested_today
    
    #Calculate portfolio value
    portfolio_value = (
        shares_spy * spy_price +
        shares_qqq * qqq_price +
        shares_vti * vti_price +
        cash
    )
    
    #Calculate earnings
    earnings = portfolio_value - total_invested
    
    #Calculate daily return
    if previous_portfolio_value is None:
        daily_return = 0
    else:
        if previous_portfolio_value != 0:
            daily_return = (portfolio_value - previous_portfolio_value) / previous_portfolio_value
        else:
            daily_return = 0
    
    previous_portfolio_value = portfolio_value
    
    #Store results for this day
    row = {
        "Date": date,
        "SPY_Price": spy_price,
        "QQQ_Price": qqq_price,
        "VTI_Price": vti_price,
        
        "SPY_MA50": spy_ma50,
        "QQQ_MA50": qqq_ma50,
        "VTI_MA50": vti_ma50,
        
        "Position_SPY": position_spy,
        "Position_QQQ": position_qqq,
        "Position_VTI": position_vti,
        
        "Shares_SPY": shares_spy,
        "Shares_QQQ": shares_qqq,
        "Shares_VTI": shares_vti,
        
        "Cash": cash,
        "Contribution": contribution,
        "Total_Invested": total_invested,
        "Portfolio_Value": portfolio_value,
        "Earnings": earnings,
        "Daily_Return": daily_return,
        
        "Strategy": "Timing",
        "Portfolio_Type": "Equal"
    }
    
    rows.append(row)

### 4. Clean and Validate Data

In [21]:
equal_timing_df = pd.DataFrame(rows)

equal_numeric_cols = [
    "SPY_Price", "QQQ_Price", "VTI_Price",
    "SPY_MA50", "QQQ_MA50", "VTI_MA50",
    "Position_SPY", "Position_QQQ", "Position_VTI",
    "Shares_SPY", "Shares_QQQ", "Shares_VTI",
    "Cash", "Contribution", "Total_Invested",
    "Portfolio_Value", "Earnings", "Daily_Return"
]

for col in equal_numeric_cols:
    equal_timing_df[col] = (
        equal_timing_df[col]
        .astype(str)
        .str.replace("`", "", regex=False)
        .str.replace("'", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    equal_timing_df[col] = pd.to_numeric(equal_timing_df[col], errors="coerce")

### 5. Export Results

In [22]:
equal_timing_df.to_csv(os.path.join(SIMULATIONS_DIR, "equal_timing.csv"), index=False)

print("equal_timing.csv created successfully")
equal_timing_df.head()

equal_timing.csv created successfully


,Date,SPY_Price,QQQ_Price,VTI_Price,SPY_MA50,QQQ_MA50,VTI_MA50,Position_SPY,Position_QQQ,Position_VTI,...,Shares_QQQ,Shares_VTI,Cash,Contribution,Total_Invested,Portfolio_Value,Earnings,Daily_Return,Strategy,Portfolio_Type
0,2001-06-15,77.995598,35.981022,36.027668,NaN,NaN,NaN,1,1,1,...,92.64143,92.521485,0.0,0,10000,10000.000000,0.000000,0.000000,Timing,Equal
1,2001-06-18,77.617950,35.499580,35.797894,NaN,NaN,NaN,1,1,1,...,92.64143,92.521485,0.0,0,10000,9917.999807,-82.000193,-0.008200,Timing,Equal
2,2001-06-19,77.957207,35.322208,35.898216,NaN,NaN,NaN,1,1,1,...,92.64143,92.521485,0.0,0,10000,9925.348770,-74.651230,0.000741,Timing,Equal
3,2001-06-20,78.366867,36.124603,36.276844,NaN,NaN,NaN,1,1,1,...,92.64143,92.521485,0.0,0,10000,10052.222823,52.222823,0.012783,Timing,Equal
4,2001-06-21,79.256599,36.614502,36.568089,NaN,NaN,NaN,1,1,1,...,92.64143,92.521485,0.0,0,10000,10162.579008,162.579008,0.010978,Timing,Equal
